# CS7180 — Solar Forecasting Training (Colab)

Trains the multi-modal Transformer on TX + CA Pecan Street data and evaluates
zero-shot transfer to NY.  Checkpoints are saved to
`s3://cs7180-final-project/checkpoints/<run_id>/` after each epoch.

**First run downloads ~500 MB from S3 automatically. Subsequent runs use local cache.**

**Runtime:** GPU > T4 recommended.  Set `Runtime → Change runtime type → T4 GPU`.

**Colab Secrets required** (left sidebar → key icon):
- `AWS_ACCESS_KEY_ID` — AWS credentials (shared by Nathan)
- `AWS_SECRET_ACCESS_KEY`

In [ ]:
# Cell 2 — Clone repo
import os
os.chdir('/content')
if not os.path.exists('/content/CS7180_final'):
    # Try git first, fall back to curl
    result = os.system('git clone https://github.com/Chaim3ra/CS7180_final /content/CS7180_final')
    if result != 0:
        os.system('curl -L https://github.com/Chaim3ra/CS7180_final/archive/refs/heads/main.zip -o /content/repo.zip')
        os.system('unzip -q /content/repo.zip -d /content')
        os.system('mv /content/CS7180_final-main /content/CS7180_final')
os.chdir('/content/CS7180_final')
!git pull origin main 2>/dev/null || echo "Could not pull latest - using downloaded version"
print(f"Working directory: {os.getcwd()}")
print(os.listdir('.'))

In [ ]:
# Cell 3 — Install dependencies
!pip install -q torch torchvision polars pyarrow boto3 lightning pyyaml numpy pvlib cdsapi xarray netCDF4

In [ ]:
# Cell 4 — Configure AWS credentials and S3 env vars
# Use Colab Secrets (left sidebar → key icon) to store credentials,
# or paste them directly (do NOT commit this notebook with real keys).
import os

try:
    from google.colab import userdata
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    print("Loaded AWS credentials from Colab Secrets")
except Exception:
    os.environ["AWS_ACCESS_KEY_ID"]     = "YOUR_KEY_HERE"
    os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_SECRET_HERE"
    print("WARNING: Using hardcoded credentials — do not commit this notebook")

os.environ["AWS_DEFAULT_REGION"]   = "us-east-2"
os.environ["S3_BUCKET"]            = "cs7180-final-project"
os.environ["S3_RAW_PREFIX"]        = "raw"
os.environ["S3_PROCESSED_PREFIX"]  = "data/processed"
os.environ["S3_CHECKPOINT_PREFIX"] = "checkpoints"

print(f"S3_BUCKET = {os.environ['S3_BUCKET']}")

In [ ]:
# Cell 5 — Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected - training will be slow on CPU")

In [ ]:
# Cell 6 — Run training
# First run downloads processed parquets (~500 MB) from S3 to local cache automatically.
# Subsequent runs reuse the local cache — no S3 reads during training.
# Checkpoints upload to S3 after each epoch.
# Smoke-test (5 epochs, 2000 windows): !python src/train.py --fast
import os
os.chdir('/content/CS7180_final')
print(f"Working directory: {os.getcwd()}")
!python src/train.py

In [ ]:
# Cell 7 — List checkpoints saved to S3
import boto3, os

bucket      = os.environ.get("S3_BUCKET", "cs7180-final-project")
ckpt_prefix = os.environ.get("S3_CHECKPOINT_PREFIX", "checkpoints")

s3   = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{ckpt_prefix}/")
objs = sorted(resp.get("Contents", []), key=lambda x: x["LastModified"], reverse=True)

print(f"Checkpoints in s3://{bucket}/{ckpt_prefix}/\n")
if objs:
    for obj in objs:
        size_mb = obj["Size"] / 1024**2
        print(f"  s3://{bucket}/{obj['Key']}  ({size_mb:.1f} MB)")
else:
    print("  No checkpoints found yet — run Cell 6 first")